# Image Summary with Gemini API

This notebook demonstrates how to upload images to Google's Gemini API and get AI-generated summaries describing the content of the pictures.

## Prerequisites
- Google Gemini API key (get it from https://makersuite.google.com/app/apikey)
- Images to analyze

## 1. Install Required Libraries

First, install the Google Generative AI package for accessing the Gemini API.

In [ ]:
!pip install google-generativeai pillow -q

## 2. Import Dependencies

Import the necessary libraries for image processing and API interaction.

In [ ]:
import google.generativeai as genai
from PIL import Image
import os
from pathlib import Path

## 3. Set Up Gemini API Authentication

Configure the Gemini API with your API key. Replace `'YOUR_API_KEY'` with your actual API key.

In [ ]:
# Configure the API key
# Option 1: Set as environment variable (recommended for security)
# os.environ['GOOGLE_API_KEY'] = 'YOUR_API_KEY'
# genai.configure(api_key=os.environ['GOOGLE_API_KEY'])

# Option 2: Direct configuration (for testing only)
API_KEY = 'YOUR_API_KEY'  # Replace with your actual API key
genai.configure(api_key=API_KEY)

# Initialize the model
model = genai.GenerativeModel('gemini-1.5-flash')

print("✓ Gemini API configured successfully!")

## 4. Load and Prepare Image

Load an image from your file system. Make sure to update the path to point to your actual image file.

In [ ]:
# Specify the path to your image
image_path = "path/to/your/image.jpg"  # Update this with your image path

# Load the image using PIL
try:
    img = Image.open(image_path)
    print(f"✓ Image loaded successfully!")
    print(f"  Size: {img.size}")
    print(f"  Format: {img.format}")
    print(f"  Mode: {img.mode}")
    
    # Display the image
    display(img)
except FileNotFoundError:
    print(f"❌ Error: Image file not found at '{image_path}'")
    print("Please update the image_path variable with the correct path to your image.")

## 5. Upload Image to Gemini API

Upload the image to the Gemini API's file service.

In [ ]:
# Upload the image file to Gemini
try:
    uploaded_file = genai.upload_file(path=image_path)
    print(f"✓ Image uploaded successfully!")
    print(f"  File URI: {uploaded_file.uri}")
    print(f"  File name: {uploaded_file.name}")
except Exception as e:
    print(f"❌ Error uploading image: {e}")

## 6. Generate Image Summary

Use the Gemini model to analyze the image and generate a detailed summary.

In [ ]:
# Create a prompt for image analysis
prompt = """
Please analyze this image and provide a detailed summary including:
1. What is shown in the image
2. Key objects, people, or elements present
3. The setting or context
4. Colors and overall mood
5. Any notable details or interesting aspects
"""

# Generate content using the uploaded image
try:
    response = model.generate_content([prompt, uploaded_file])
    summary = response.text
    print("✓ Summary generated successfully!\n")
except Exception as e:
    print(f"❌ Error generating summary: {e}")
    summary = None

## 7. Display Results

Display the image alongside its AI-generated summary.

In [ ]:
if summary:
    print("=" * 80)
    print("IMAGE SUMMARY")
    print("=" * 80)
    print(summary)
    print("=" * 80)
    
    # Display the image again for reference
    print("\nOriginal Image:")
    display(img)

## 8. Process Multiple Images

Create a function to batch process multiple images and generate summaries for each one.

In [ ]:
def analyze_image(image_path, custom_prompt=None):
    """
    Analyze a single image and return its summary.
    
    Args:
        image_path: Path to the image file
        custom_prompt: Optional custom prompt for analysis
    
    Returns:
        Dictionary with image info and summary
    """
    default_prompt = "Please provide a detailed description of this image."
    prompt = custom_prompt if custom_prompt else default_prompt
    
    try:
        # Load image
        img = Image.open(image_path)
        
        # Upload to Gemini
        uploaded_file = genai.upload_file(path=image_path)
        
        # Generate summary
        response = model.generate_content([prompt, uploaded_file])
        
        return {
            'path': image_path,
            'size': img.size,
            'format': img.format,
            'summary': response.text,
            'status': 'success'
        }
    except Exception as e:
        return {
            'path': image_path,
            'status': 'error',
            'error': str(e)
        }


def analyze_multiple_images(image_paths, custom_prompt=None):
    """
    Analyze multiple images at once.
    
    Args:
        image_paths: List of paths to image files
        custom_prompt: Optional custom prompt for all images
    
    Returns:
        List of dictionaries with results for each image
    """
    results = []
    
    for i, path in enumerate(image_paths, 1):
        print(f"\n[{i}/{len(image_paths)}] Processing: {path}")
        result = analyze_image(path, custom_prompt)
        results.append(result)
        
        if result['status'] == 'success':
            print(f"✓ Success")
        else:
            print(f"❌ Error: {result['error']}")
    
    return results

### Example: Batch Process Multiple Images

Uncomment and run the code below to process multiple images at once.

In [ ]:
# Example usage - uncomment and update paths
"""
# List of image paths to analyze
image_list = [
    "path/to/image1.jpg",
    "path/to/image2.png",
    "path/to/image3.jpg"
]

# Optional: Create a custom prompt
custom_prompt = "Describe this image in detail, focusing on colors and composition."

# Analyze all images
results = analyze_multiple_images(image_list, custom_prompt)

# Display results
for i, result in enumerate(results, 1):
    print(f"\n{'='*80}")
    print(f"IMAGE {i}: {result['path']}")
    print('='*80)
    if result['status'] == 'success':
        print(f"Size: {result['size']}")
        print(f"Format: {result['format']}")
        print(f"\nSummary:\n{result['summary']}")
    else:
        print(f"Error: {result['error']}")
"""

print("💡 Tip: Uncomment the code above and update the image paths to batch process multiple images.")

## Tips & Best Practices

### Getting Your API Key
1. Visit https://makersuite.google.com/app/apikey
2. Sign in with your Google account
3. Create a new API key
4. Copy the key and paste it in the configuration cell above

### Supported Image Formats
- JPEG (.jpg, .jpeg)
- PNG (.png)
- WebP (.webp)
- HEIC (.heic)
- HEIF (.heif)

### Cost Considerations
- Gemini API has usage limits and pricing
- Check current pricing at https://ai.google.dev/pricing
- The free tier includes generous limits for testing

### Clean Up (Optional)
Files uploaded to Gemini API are automatically deleted after a period of time, but you can manually delete them if needed.

In [ ]:
# Optional: List and clean up uploaded files
"""
# List all uploaded files
files = genai.list_files()
print("Uploaded files:")
for file in files:
    print(f"  - {file.name} (uploaded: {file.create_time})")

# Delete a specific file (uncomment to use)
# genai.delete_file(uploaded_file.name)
# print("File deleted successfully!")
"""

print("✓ Notebook complete! Update the API key and image path, then run the cells to analyze your images.")